In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

In [2]:
data = r"C:\Users\rajeshkumar.t\Downloads\6025ac4bfe90ad7c55f5526d6e9af44a.csv"
df = pd.read_csv(data)


In [ ]:
cols_to_drop = [
    'status', 'id', 'order_item_id', 'order_external_id', 'checkout_id', 
    'product_title', 'tracking_id', 'shipping_address_id', 'pincode', 
    'order_id', 'sku', 'unit_creation_date_time'
]
    
X = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
y= (df['status']=="DELIVERED").astype(int)

cat_col = X.select_dtypes(include=['object']).columns
num_col = X.select_dtypes(exclude=['object']).columns

X[cat_col] = X[cat_col].astype(str)

preprocessor = ColumnTransformer(
    transformers= [
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value = 'missing')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
        ]), cat_col),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_col)
    ]
)

model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators= 20,max_depth=10, random_state=42, n_jobs=-1))
])
                
X_train, X_test, y_train, y_test=train_test_split(X,y, test_size=0.2, random_state=42)
print("start training")
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))